# 04 - DBSCAN Clustering

**Purpose**: Density-based clustering with automatic outlier detection

**Features**:
- Interactive parameter tuning (eps, min_samples)
- Automatic outlier detection
- Visualization of cluster density
- Comparison with other algorithms

---

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

from notebook_utils import setup_notebook, plot_cluster_distribution

%matplotlib inline
print("✓ Imports loaded")

In [ ]:
# Load state
cfg, state = setup_notebook("DBSCAN Clustering", "germany")

df_latest = state.load('df_latest')
feature_cols = state.load('feature_cols')
kmeans_scaler = state.load('kmeans_scaler')

X_scaled = kmeans_scaler.transform(df_latest[feature_cols].fillna(df_latest[feature_cols].median()))

## Parameter Selection: K-Distance Plot

In [ ]:
# K-distance plot to determine eps
MIN_SAMPLES = 5  # Rule of thumb: 2 * n_features or use domain knowledge

# Calculate distances to k-th nearest neighbor
neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES)
neighbors.fit(X_scaled)
distances, indices = neighbors.kneighbors(X_scaled)

# Sort distances
k_distances = np.sort(distances[:, -1])[::-1]

# Plot k-distance
plt.figure(figsize=(12, 6))
plt.plot(k_distances)
plt.xlabel('Data Points (sorted)', fontsize=12)
plt.ylabel(f'{MIN_SAMPLES}-th Nearest Neighbor Distance', fontsize=12)
plt.title('K-Distance Plot (Elbow indicates optimal eps)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Suggest eps value
suggested_eps = np.percentile(k_distances, 95)
print(f"\n💡 Suggested eps (95th percentile): {suggested_eps:.3f}")

## DBSCAN Configuration & Execution

In [ ]:
# Configure DBSCAN
EPS = suggested_eps  # Adjust based on k-distance plot
MIN_SAMPLES = 5

print(f"DBSCAN Configuration: eps={EPS:.3f}, min_samples={MIN_SAMPLES}")

# Fit DBSCAN
dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Count clusters and outliers
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_outliers = list(dbscan_labels).count(-1)

print(f"\n✓ DBSCAN complete")
print(f"  Clusters found: {n_clusters}")
print(f"  Outliers: {n_outliers} ({n_outliers/len(dbscan_labels)*100:.1f}%)")

## Cluster Analysis

In [ ]:
# Add labels to dataframe
df_result = df_latest.copy()
df_result['cluster_dbscan'] = dbscan_labels

# Visualize distribution (excluding outliers -1)
if n_clusters > 0:
    df_clustered = df_result[df_result['cluster_dbscan'] != -1]
    cluster_counts = plot_cluster_distribution(df_clustered, 'cluster_dbscan', 'DBSCAN Clusters')
    
    # Calculate silhouette (excluding outliers)
    if len(df_clustered) > 0 and n_clusters > 1:
        mask = dbscan_labels != -1
        silhouette = silhouette_score(X_scaled[mask], dbscan_labels[mask])
        print(f"\nSilhouette Score (excluding outliers): {silhouette:.3f}")
else:
    print("\n⚠️  No clusters found. Try adjusting eps or min_samples.")

In [ ]:
# Outlier analysis
if n_outliers > 0:
    print(f"\n📊 Outlier Analysis:")
    print(f"  Total outliers: {n_outliers}")
    print(f"  Percentage: {n_outliers/len(dbscan_labels)*100:.1f}%")
    
    # Show sample outliers
    outliers = df_result[df_result['cluster_dbscan'] == -1]
    print(f"\n  Sample outliers:")
    display(outliers.head())

## Parameter Grid Search (Optional)

In [ ]:
# Try different parameter combinations
eps_values = np.linspace(EPS * 0.5, EPS * 2, 10)
min_samples_values = [3, 5, 10]

results = []

for eps in eps_values:
    for min_samp in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=min_samp)
        labels = db.fit_predict(X_scaled)
        
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        n_out = list(labels).count(-1)
        
        results.append({
            'eps': eps,
            'min_samples': min_samp,
            'n_clusters': n_clust,
            'n_outliers': n_out,
            'outlier_pct': n_out / len(labels) * 100
        })

df_grid = pd.DataFrame(results)

print("\n📊 Parameter Grid Search Results (top 10):")
display(df_grid.nlargest(10, 'n_clusters'))

## Save Results

In [ ]:
state.save('dbscan_labels', dbscan_labels)
state.save('dbscan_results', df_result)
state.save('dbscan_metrics', {
    'eps': EPS,
    'min_samples': MIN_SAMPLES,
    'n_clusters': n_clusters,
    'n_outliers': n_outliers
})

print("\n✓ Results saved to state")
print("\n📝 Next: 05_Algorithm_Comparison.ipynb")